# 02 — Exploratory Data Analysis

**Project:** Store-Level Retail Demand Forecasting (Rossmann Store Sales)
**Notebook goal:** build the empirical case for every modeling choice made later in this project, specifically:

1. Why the target is modeled as `log1p(Sales)` rather than raw `Sales` (distribution shape).
2. What the trend/seasonal structure of daily sales looks like, using an STL decomposition that is robust to promo-driven outlier spikes (period = 7, weekly seasonality).
3. Which exogenous signals (promo, holidays, store type, assortment, competition distance) visibly move Sales and are therefore worth engineering into features in notebook 03.
4. A leakage trap: `Customers` is almost perfectly correlated with `Sales` but is not available in `test.csv`, so it must never enter the feature set.

This notebook reads the already-cleaned, already-merged frame produced by notebook 01 (`data/interim/train_merged_clean.parquet`) — it does no cleaning itself, only analysis and visualization. All figures referenced in the written report are saved under `reports/figures/`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_io import load_parquet, read_raw_csvs
from src.viz import decomposition_plot, promo_effect_plot, save_fig

# A single sequential hue used consistently for every "plain magnitude" bar
# chart in this notebook (day-of-week, missing-data, competition-distance
# bins), so color never has to be re-interpreted chart to chart. Categorical
# comparisons (StoreType, Assortment) get a colorblind-safe palette instead,
# and the correlation heatmap gets a diverging palette centered on 0 -- each
# chart's color encodes the kind of comparison it is making.
SEQ_COLOR = "#4C72B0"

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Load the cleaned, merged frame

This is the exact output of notebook 01's `clean_pipeline` (train + store merged, `StateHoliday` encoded, competition/promo2 fields imputed). Note that `Open == 0` rows are intentionally still present — notebook 01 keeps them so that per-store date indices stay continuous for the lag/rolling features built in notebook 03. For most of the plots below we will explicitly restrict to `Open == 1`, since a closed store's `Sales` is trivially 0 and would otherwise dominate every distribution.

In [ ]:
df = load_parquet("../data/interim/train_merged_clean.parquet")
print(df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Stores:", df["Store"].nunique())
df.head()

In [ ]:
open_df = df.loc[df.Open == 1].copy()
print(f"Open==1 rows: {len(open_df):,} of {len(df):,} total ({len(open_df) / len(df):.1%})")

## Sales distribution and the case for `log1p`

Retail sales are a classic right-skewed quantity: most days cluster around a typical volume, but promo days, pre-holiday rushes, and strong stores pull a long tail toward large values. A model trained directly on raw `Sales` lets that tail dominate the loss (squared-error losses in particular are quadratically sensitive to a handful of huge days), which is exactly why the target for every model in this project is `log1p(Sales)`, not `Sales`. The plots and skew statistics below are the evidence for that choice.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(open_df["Sales"], bins=60, color=SEQ_COLOR, edgecolor="white")
axes[0].set_title("Sales (Open == 1), raw")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Count")

axes[1].hist(np.log1p(open_df["Sales"]), bins=60, color=SEQ_COLOR, edgecolor="white")
axes[1].set_title("log1p(Sales) (Open == 1)")
axes[1].set_xlabel("log1p(Sales)")
axes[1].set_ylabel("Count")

fig.tight_layout()
plt.show()

raw_skew = open_df["Sales"].skew()
log_skew = np.log1p(open_df["Sales"]).skew()
print(f"Sales skew (Open==1):        {raw_skew:.2f}")
print(f"log1p(Sales) skew (Open==1): {log_skew:.2f}")

Those skew numbers look wrong at first glance — the log1p histogram looks *more* skewed (-7.25) than the raw one (-0.39), which is the opposite of what log-transforming a right-skewed distribution should do. The two histograms above explain why: both show a second, isolated spike at (or near) zero that sits far outside the main bell of the distribution. On the raw scale that spike is just one bin among thousands at the low end of a spread-out axis, so it barely dents the skew. On the log1p scale, `log1p(0) = 0` sits roughly 8 log-units away from the bulk of the distribution (which clusters around `log1p(6000) ≈ 8.7`), so those same rows become extreme outliers that single-handedly flip the sign of the skew statistic. This is worth chasing down rather than ignoring, so the next cell isolates exactly what those zero-Sales-while-open rows are.

In [ ]:
zero_sales_open = open_df.loc[open_df["Sales"] == 0]
n_zero = len(zero_sales_open)
print(f"Open==1 rows with Sales==0: {n_zero} ({n_zero / len(open_df):.2%} of open rows)")
print("DayOfWeek breakdown of these rows:", zero_sales_open["DayOfWeek"].value_counts().to_dict())

nonzero = open_df.loc[open_df["Sales"] > 0]
raw_skew_nz = nonzero["Sales"].skew()
log_skew_nz = np.log1p(nonzero["Sales"]).skew()
print(f"\nExcluding those rows -> Sales skew: {raw_skew_nz:.2f}, log1p(Sales) skew: {log_skew_nz:.2f}")

That confirms it: **all 197 `Open == 1` / `Sales == 0` rows fall on `DayOfWeek == 7` (Sunday)**. Every store in this sample is normally closed on Sundays (see the day-of-week section below, where Sunday's mean Sales is driven to essentially zero by the same rows), so these 197 records look like an `Open` flag that was mis-set at the snapshot/export stage in the original Kaggle extract rather than a real day of zero demand — the store almost certainly was not actually trading. This is a known quirk of this dataset, not a bug in notebook 01's cleaning: `Open == 0` rows are kept by design, but a handful of rows are flagged open when they evidently were not.

Once those 197 rows are set aside, the picture matches the textbook story cleanly: raw Sales is right-skewed (skew ≈ 0.82, a visible long tail toward large values), and `log1p` pulls that tail in sharply (skew ≈ 0.26, close to symmetric). That is the real motivation for modeling `log1p(Sales)`. The mislabeled-Sunday rows are a separate data-quality wrinkle; `features.build_feature_matrix` does not drop them automatically (its `is_trainable` flag only checks `Open == 1` and completeness of the lag/rolling columns), so they will still flow into notebook 03's training set — worth remembering if residuals on Sundays look unusually large later.

## Trend/seasonal decomposition (STL)

To see the trend and seasonal structure separately from day-to-day noise, we decompose the **daily total Sales across all open stores** using STL (Seasonal-Trend decomposition using LOESS, via `src.viz.decomposition_plot`, which wraps `statsmodels.tsa.seasonal.STL`). Two choices matter here:

- **`robust=True`** (set inside `decomposition_plot`): STL's robust mode down-weights outlier days when fitting the trend/seasonal components, instead of letting them distort the fit. That matters a lot for retail data, where promo spikes and the pre-holiday rush produce days that are genuine outliers relative to the local trend — without robust fitting those spikes would leak into the trend component and make it look noisier than the underlying trend actually is.
- **`period=7`**: daily retail sales have a strong, well-defined weekly cycle (weekday vs. weekend shopping patterns), so a 7-day period is the obvious and correct seasonal period to decompose against.

What we deliberately do *not* do is run STL with `period=365` to pull out an annual cycle. This dataset spans only about 2.5 years (see the date range printed above), so a period-365 decomposition would have only ~2.5 full cycles to estimate the seasonal component from — far too few for STL's LOESS smoothing to produce a reliable annual seasonal curve. Instead, annual seasonality is examined qualitatively further below by resampling to monthly means and just looking at the shape.

In [ ]:
daily_total = df[df.Open == 1].groupby("Date")["Sales"].sum()
print(f"daily_total: {len(daily_total)} days, {daily_total.index.min().date()} to {daily_total.index.max().date()}")

full_range = pd.date_range(daily_total.index.min(), daily_total.index.max(), freq="D")
n_missing_days = len(full_range.difference(daily_total.index))
print(f"Calendar days with zero open stores anywhere (absent from this series): {n_missing_days} of {len(full_range)}")

fig, result = decomposition_plot(daily_total, period=7)
save_fig(fig, "../reports/figures/decomposition_weekly.png")
plt.show()

A couple of things worth flagging about this series before reading the decomposition:

- **It has small gaps.** 49 of the 942 calendar days in range have *no* open stores at all (mostly region-wide holidays where every store closes) and are simply absent from `daily_total` rather than present with a 0. STL has no notion of calendar dates here — it treats the series positionally and assumes every 7th point is the same weekday. Around each gap that assumption is briefly off by a day, which shows up as small local phase noise in the seasonal component, but with only 49 gaps spread across 893 observed days it does not materially change the overall weekly pattern.
- **Reading the four panels:** the *Trend* panel rises from roughly 68k to 110k in total daily sales across the window — the stores in this sample are growing, not flat. The *Seasonal* panel shows a clean, large-amplitude weekly oscillation (roughly ±40–90k around the trend), confirming day-of-week is the dominant seasonal driver here, consistent with the day-of-week barplot below. The *Residual* panel is centered near zero with sharp, isolated spikes — those are exactly the promo/outlier days STL's robust fitting intentionally pulled out of the trend and seasonal components instead of smoothing over.

In [ ]:
monthly_total = daily_total.resample("MS").mean()

fig, ax = plt.subplots(figsize=(10, 4))
monthly_total.plot(ax=ax, marker="o", color=SEQ_COLOR)
ax.set_title("Monthly mean daily Sales (qualitative annual view)")
ax.set_ylabel("Mean daily Sales (all open stores)")
ax.set_xlabel("Month")
fig.tight_layout()
plt.show()

This is the qualitative substitute for a formal annual decomposition. With only 2.5 cycles of data we cannot fit a reliable period-365 seasonal component, but the monthly means still show a usable pattern: there is a gentle multi-year upward drift (consistent with the STL trend panel above) layered on top of what looks like a recurring late-year bump around autumn/holiday months in each of the two full calendar years present. Treat this as a directional read, not a decomposed seasonal estimate — notebook 03's calendar features (`Month`, `Quarter`, `WeekOfYear`, `is_day_before_holiday`, etc.) are how a model gets to use this signal without needing a formal STL fit to support it.

## Day-of-week seasonality

STL's seasonal panel already told us weekly seasonality is strong; this section quantifies it directly as mean Sales per `DayOfWeek` on the `Open == 1` subset.

In [ ]:
dow_means = open_df.groupby("DayOfWeek")["Sales"].mean()

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=dow_means.index, y=dow_means.values, color=SEQ_COLOR, ax=ax)
ax.set_xlabel("DayOfWeek (1 = Monday ... 7 = Sunday)")
ax.set_ylabel("Mean Sales")
ax.set_title("Mean Sales by Day of Week (Open == 1)")
fig.tight_layout()
save_fig(fig, "../reports/figures/dow_seasonality.png")
plt.show()

print(dow_means.round(1))

Monday through Friday sit in a fairly tight band (~5,400–6,300), Saturday (6) is the clear weekly peak at ~7,830 — consistent with weekend shopping behavior — and Sunday (7) reads as essentially zero. That zero is not a real "Sunday demand collapse": it is exactly the 197 mislabeled `Open==1`/`Sales==0` rows identified in the sales-distribution section above, which are the *only* `Open==1` Sunday records in this sample. In other words, every store in this sample is genuinely closed on Sundays, and the small number of Sunday rows flagged open in the raw data all recorded zero sales — the two findings are the same anomaly seen from two angles.

## Promo effect

`src.viz.promo_effect_plot` compares mean Sales on promo vs. non-promo days directly.

In [ ]:
fig = promo_effect_plot(open_df)
save_fig(fig, "../reports/figures/promo_effect.png")
plt.show()

promo_mean = open_df.loc[open_df.Promo == 1, "Sales"].mean()
no_promo_mean = open_df.loc[open_df.Promo == 0, "Sales"].mean()
promo_lift_pct = (promo_mean / no_promo_mean - 1) * 100
print(f"Mean Sales, Promo=1: {promo_mean:,.1f}")
print(f"Mean Sales, Promo=0: {no_promo_mean:,.1f}")
print(f"Promo lift: {promo_lift_pct:.2f}%")

Running a promo lifts mean Sales by roughly 19% on an open, non-promo baseline of ~5,745. That is a large, consistent effect — easily the strongest single lever in this dataset other than day-of-week — and it is exactly why `Promo` is part of `NUMERIC_FEATURE_COLS` and why `promo_last_7_days_count` is engineered as a rolling feature in notebook 03 (recent promo cadence, not just today's flag, carries information about demand).

## Holiday effect

Two separate holiday-ish signals exist in this data: `state_holiday_type` (public/bank holidays, encoded from `StateHoliday`) and `SchoolHoliday` (school break days). We look at each against Sales on the `Open == 1` subset.

In [ ]:
holiday_open_crosstab = df.groupby(["state_holiday_type", "Open"], observed=True).size().unstack(fill_value=0)
print(holiday_open_crosstab)

This crosstab matters for reading the boxplot below correctly: **every single state-holiday row (`state_holiday_type == "a"`) in this sample has `Open == 0`** — stores in this dataset close entirely on public holidays rather than staying open with reduced demand. That means restricting to `Open == 1` leaves only the `"0"` (no holiday) category with any data at all, so a same-axis boxplot comparison is necessarily degenerate. We plot it anyway, as specified, precisely to make that point visible rather than hide it — the real holiday effect in this dataset operates almost entirely through forced closure (`Sales = 0` because the store didn't open), not through a demand shift on days the store happened to stay open.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.boxplot(
    data=open_df, x="state_holiday_type", y="Sales",
    hue="state_holiday_type", palette="colorblind", legend=False, ax=ax,
)
ax.set_title("Sales by state_holiday_type (Open == 1 subset)")
fig.tight_layout()
plt.show()

school_means = open_df.groupby("SchoolHoliday")["Sales"].mean()
school_lift_pct = (school_means.loc[1] / school_means.loc[0] - 1) * 100
print(school_means.round(1))
print(f"SchoolHoliday lift: {school_lift_pct:.2f}%")

As expected, the state-holiday boxplot is a single box — there is nothing to compare within the open-store subset. `SchoolHoliday`, in contrast, does have variation among open stores and shows only a small, ~1.8% lift in mean Sales on school-holiday days. That is a far weaker signal than `Promo` (~19%) or day-of-week, but it is a real and consistent one, which is why `SchoolHoliday` stays in `NUMERIC_FEATURE_COLS` while the main lever from holidays flows through `is_state_holiday` and the day-before/day-after holiday calendar flags instead of through `state_holiday_type` directly.

## Store-type and assortment differences

`StoreType` (a/b/c/d) and `Assortment` (a=basic, b=extra, c=extended) are store-level attributes merged in from `store.csv`. We compare their Sales distributions on the `Open == 1` subset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.boxplot(
    data=open_df, x="StoreType", y="Sales", hue="StoreType",
    order=sorted(open_df["StoreType"].unique()), palette="colorblind", legend=False, ax=axes[0],
)
axes[0].set_title("Sales by StoreType (Open == 1)")

sns.boxplot(
    data=open_df, x="Assortment", y="Sales", hue="Assortment",
    order=sorted(open_df["Assortment"].unique()), palette="colorblind", legend=False, ax=axes[1],
)
axes[1].set_title("Sales by Assortment (Open == 1)")

fig.tight_layout()
save_fig(fig, "../reports/figures/storetype_boxplot.png")
plt.show()

print(open_df.groupby("StoreType")["Sales"].mean().round(1).sort_values(ascending=False))
print()
print(open_df.groupby("Assortment")["Sales"].mean().round(1).sort_values(ascending=False))

`StoreType` separates cleanly: type `c` (~6,416 mean) and `d` (~6,168) clearly outsell `a` (~5,865) and especially `b` (~5,448), which is also the type with the widest spread and heaviest upper-tail outliers in the boxplot. `Assortment` shows a smaller but still visible gap, with `b` ("extra" assortment, ~6,248) ahead of `c` ("extended", ~5,972) and `a` ("basic", ~5,633) — more product variety is associated with more Sales, as expected. Both are categorical and both belong in `CATEGORICAL_FEATURE_COLS` for exactly this reason: they are store-level, not time-varying, so a tree model or a one-hot/category-encoded linear model can use them directly as a per-store baseline shifter.

## CompetitionDistance vs. mean Sales

`CompetitionDistance` (meters to the nearest competitor) is binned into 5 equal-width buckets with `pd.cut` and compared against mean Sales.

In [ ]:
open_df["competition_distance_bin"] = pd.cut(open_df["CompetitionDistance"], bins=5)
comp_means = open_df.groupby("competition_distance_bin", observed=False)["Sales"].mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
comp_means.plot(kind="bar", color=SEQ_COLOR, ax=ax)
ax.set_ylabel("Mean Sales")
ax.set_xlabel("CompetitionDistance bin")
ax.set_title("Mean Sales by CompetitionDistance bin (Open == 1)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
plt.show()

print(comp_means)

Two of the five bins come back empty — this sample has only 15 stores and just 6 distinct `CompetitionDistance` values among them (50m up to 8,000m), so an equal-width 5-bin split leaves gaps in the middle of the range where no store happens to fall. Within the three bins that do have data, mean Sales barely moves (roughly 5,800–6,120) and doesn't move monotonically with distance — if anything, the nearest-competitor bin has the *highest* mean Sales, not the lowest. That is a genuinely weak, non-monotonic relationship in this sample, which makes sense: raw distance to the nearest competitor says nothing about that competitor's size, price positioning, or whether it's actually drawing the same customers, so it's a noisy proxy for competitive pressure at best. `CompetitionDistance` is still kept in `NUMERIC_FEATURE_COLS` (gradient-boosted trees in particular can pick up non-monotonic, threshold-like effects that a bar chart of 5 wide bins would wash out), but this plot is a reason not to expect it to be a dominant driver on its own.

## Missing data in the raw store attributes

Notebook 01 imputes several `store.csv` fields (median `CompetitionDistance`, derived `competition_open_days`/`has_competitor_open` flags, `promo2_active`). Before trusting those imputations, it's worth re-confirming exactly how much was actually missing in the **raw** file — reloaded here directly via `read_raw_csvs`, independent of anything notebook 01 already did to it.

In [ ]:
_, _, store_raw = read_raw_csvs("../data/raw")
missing_pct = (store_raw.isna().mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
missing_pct.plot(kind="bar", color=SEQ_COLOR, ax=ax)
ax.set_ylabel("% missing")
ax.set_title("Missing values in raw store.csv")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
plt.show()

print(missing_pct.round(2))

The Promo2-related fields (`Promo2SinceWeek`, `Promo2SinceYear`, `PromoInterval`) are missing for ~47% of stores — exactly the stores where `Promo2 == 0` (no ongoing promo program), so that "missingness" is structural, not a data problem, and `derive_promo2_active` handles it correctly by only computing an active-promo flag when the fields are present. `CompetitionOpenSinceYear`/`Month` are missing for 40%/27% of stores, and bare `CompetitionDistance` for 13% — these are genuine unknowns (we don't know when, or in `CompetitionDistance`'s case whether, a competitor opened nearby), which is exactly why notebook 01 adds explicit `competition_distance_missing` and `has_competition_since_info` indicator columns rather than silently imputing and hoping a median value doesn't bias the model. Both indicator columns are present in `NUMERIC_FEATURE_COLS`, so a model can learn a different behavior for "we don't know" vs. "we know and it's far away."

## Correlation heatmap — and the Customers leakage trap

For this correlation check we deliberately use the **full** `df`, not the `Open == 1` subset: `is_state_holiday` is constant (always 0) once you restrict to open stores, since this dataset's stores close entirely on state holidays (same finding as the holiday-effect section above), which would make its correlation with everything else undefined. Keeping the closed-store rows in — where `Sales` is 0 by construction — is exactly what lets `is_state_holiday` (and to a lesser extent `Promo`) show a real relationship with `Sales` here.

In [ ]:
corr_cols = ["Sales", "Customers", "Promo", "SchoolHoliday", "CompetitionDistance", "is_state_holiday"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title("Correlation matrix (full df, includes Open==0 rows)")
fig.tight_layout()
plt.show()

print(corr.round(3))

`is_state_holiday` correlates at about -0.30 with `Sales` in this full-data view (holidays push Sales toward 0, mostly via closure), and `Promo` at about +0.38 — both sensible given the sections above. `CompetitionDistance` and `SchoolHoliday` are both close to 0, consistent with the weak effects already found for each. One number here dwarfs everything else, though, and deserves its own callout rather than just a cell in the heatmap.

In [ ]:
print(df[["Sales", "Customers"]].corr())

**Leakage trap.** `Sales` and `Customers` correlate at **r ≈ 0.966** — by far the strongest relationship in this entire dataset, stronger than any exogenous driver found above. That makes intuitive sense (more customers in the store on a given day essentially *is* more sales), but it also makes `Customers` a textbook example of target leakage: it is a **contemporaneous, same-day outcome** of the exact same store visit that produces `Sales`, not a predictor known in advance, and critically **it does not exist in `test.csv`** at all. A model trained with `Customers` in its feature set would learn to lean almost entirely on this one near-perfect proxy, would show unrealistically good validation metrics (because validation still has `Customers` available), and would then have no equivalent signal to fall back on at actual forecast/inference time, where `Customers` is simply unknown.

This is exactly why `Customers` is **absent from both `NUMERIC_FEATURE_COLS` and `CATEGORICAL_FEATURE_COLS`** in `src/features.py` — not an oversight, but a deliberate exclusion driven directly by this correlation check. Every feature that does make it into those two lists (lags/rolling stats of `Sales` itself, calendar fields, `Promo`, store attributes, competition/promo2 derived fields) is either known in advance or derivable purely from the store's own sales history up to, but not including, the day being predicted.

## Summary — what this EDA hands off to notebook 03

- **Target transform:** model `log1p(Sales)`. Raw `Sales` (Open==1) is right-skewed (skew ≈ 0.82 once the 197 mislabeled-Sunday zero rows are set aside); `log1p` compresses that to near-symmetric (skew ≈ 0.26).
- **Data quirk to remember:** 197 `Open==1` rows with `Sales==0` are *all* on Sundays, the one day of the week every store in this sample is structurally closed — almost certainly a mis-set `Open` flag rather than real zero-demand trading days. `features.build_feature_matrix`'s `is_trainable` flag does not filter these out, so they remain in the training set.
- **Seasonality:** weekly seasonality dominates (confirmed by both the STL seasonal panel and the day-of-week barplot — Saturday is the weekly peak); with only ~2.5 years of history, annual seasonality can only be read qualitatively (monthly resample), not formally decomposed.
- **Strongest engineered-feature drivers found here:** `Promo` (~19% lift), `StoreType`/`Assortment` (clear, separable group differences), state holidays (operate almost entirely through forced closure, captured by `is_state_holiday` and the before/after-holiday calendar flags rather than `state_holiday_type` itself once stores are open).
- **Weak drivers:** `SchoolHoliday` (~1.8% lift) and `CompetitionDistance` (small, non-monotonic in this sample) — both still worth including as features for a tree-based model to exploit non-linearly, but neither should be expected to move metrics much on its own.
- **Imputation sanity check:** the missingness in raw `store.csv` (13%–47% across competition/promo2 fields) is largely structural (e.g. `Promo2`-inactive stores), and notebook 01's `competition_distance_missing`/`has_competition_since_info` indicator columns correctly let a model distinguish "unknown" from "known and far away."
- **Leakage trap:** `Customers` correlates with `Sales` at r ≈ 0.966 but is absent from `test.csv` — confirmed excluded from both `NUMERIC_FEATURE_COLS` and `CATEGORICAL_FEATURE_COLS`.

Notebook 03 picks up from here to build the full engineered feature matrix (`build_feature_matrix`) on top of these findings.